# ABLH ceilometer single-date workbench

This notebook estimates the **atmospheric boundary layer height (ABLH)** with two methods:

- **WCT** (`wct`): detects strong vertical gradients in the backscatter signal.
- **Temporal variance** (`temporal_variance`): detects temporal variability associated with the boundary layer.

The main idea is to keep the role of each data source clear:

- **Raw data**: used for the ABLH calculation because it preserves the original temporal resolution.
- **Cloudnet data**: used mainly as a noise mask, because its filtering is useful even though its 5-minute smoothing can be problematic for methods that depend on fine temporal variability.

Run the cells in order. The last cell processes one date.

## 1. Imports

This cell loads the required libraries. If an import fails, it usually means that the active Python/Jupyter environment is missing a package.

In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from lidarpy.retrieval.ablh import detect_ablh

## 2. Function Definitions

These functions do four things:

1. Open NetCDF files.
2. Prepare the backscatter variable with dimensions `(time, range)`.
3. Apply the Cloudnet mask to the raw data.
4. Run the two ABLH methods and save results/figures.

For normal use, you should not need to edit these functions. Usually, only the parameters in section 3 need to change.

In [ ]:
METHODS = ("wct", "temporal_variance")

def load_backscatter_for_ablh(
    input_path: Path,
    *,
    variable: str,
    time_frequency: str | None = None,
) -> xr.Dataset:
    """Load one backscatter variable and return it as an ABLH-ready Dataset.

    Parameters
    ----------
    input_path
        NetCDF file to open.
    variable
        Name of the variable to use. For example, raw files often use
        ``beta_raw`` and Cloudnet files often use ``beta``.
    time_frequency
        Optional resampling frequency, e.g. ``"5min"``. Use ``None`` to keep
        the original temporal resolution.
    """
    input_path = Path(input_path).expanduser()
    if not input_path.exists():
        raise FileNotFoundError(f"Input file not found: {input_path}")

    dataset = xr.open_dataset(input_path)
    if variable not in dataset:
        available = ", ".join(dataset.data_vars)
        raise KeyError(
            f"Variable {variable!r} not found in {input_path.name}. "
            f"Available variables: {available}"
        )

    dataset = dataset.sortby("time")

    # Some files can contain duplicated timestamps. Grouping by time gives a
    # clean, unique time axis before optional resampling/reindexing.
    if "time" in dataset.indexes and not dataset.indexes["time"].is_unique:
        dataset = dataset.groupby("time").median()

    backscatter = dataset[variable].transpose("time", "range")
    if time_frequency is not None:
        backscatter = backscatter.resample(time=time_frequency).median(skipna=True)

    output = xr.Dataset({variable: backscatter})
    if "height" in dataset:
        output["height"] = dataset["height"]
    output.attrs.update(dataset.attrs)
    return output


def apply_cloudnet_noise_mask(
    raw_dataset: xr.Dataset,
    cloudnet_dataset: xr.Dataset,
    *,
    raw_variable: str = "beta_raw",
    cloudnet_variable: str = "beta",
) -> xr.Dataset:
    """Mask raw data where Cloudnet has no valid signal.

    Cloudnet has a good noise filter. We use its NaN pattern as a quality mask,
    but we keep the raw values and raw time resolution for the actual ABLH
    calculation.
    """
    masked = raw_dataset.copy()
    cloudnet_nan_mask = np.isnan(cloudnet_dataset[cloudnet_variable])

    # Cloudnet and raw files may not share exactly the same time/range grid.
    # Nearest-neighbour reindexing transfers the Cloudnet mask onto the raw grid.
    cloudnet_nan_mask = cloudnet_nan_mask.reindex_like(
        masked[raw_variable],
        method="nearest",
    )
    masked[raw_variable] = masked[raw_variable].where(~cloudnet_nan_mask)
    return masked


def _log10_positive(values: np.ndarray) -> np.ndarray:
    """Return log10(values), preserving NaNs and avoiding log of non-positive values."""
    positive = np.where(values > 0, values, np.nan)
    finite_positive = np.isfinite(positive)

    if not finite_positive.any():
        return np.full_like(values, np.nan, dtype=float)

    floor = np.nanpercentile(positive[finite_positive], 1)
    if not np.isfinite(floor) or floor <= 0:
        floor = np.nanmin(positive[finite_positive])

    return np.log10(np.maximum(positive, floor))


def plot_ablh_quicklook(
    *,
    dataset: xr.Dataset,
    ablh_results: dict[str, xr.Dataset],
    variable: str,
    min_range: float,
    max_range: float,
    output_path: Path,
) -> None:
    """Plot backscatter plus the ABLH retrieved by each method."""
    backscatter = dataset[variable].transpose("time", "range")
    ranges = backscatter["range"].values.astype(float)
    range_mask = (ranges >= min_range) & (ranges <= max_range)

    plot_signal = _log10_positive(backscatter.values[:, range_mask])

    fig, ax = plt.subplots(figsize=(13, 5), constrained_layout=True)
    mesh = ax.pcolormesh(
        backscatter["time"].values,
        ranges[range_mask] / 1000.0,
        plot_signal.T,
        shading="auto",
        cmap="viridis",
    )

    colors = {"wct": "white", "temporal_variance": "tab:red"}
    labels = {"wct": "ABLH WCT", "temporal_variance": "ABLH temporal variance"}

    for method, result in ablh_results.items():
        ax.plot(
            result["time"].values,
            result["ablh"].values / 1000.0,
            color=colors.get(method, "tab:orange"),
            lw=2.0 if method == "wct" else 1.6,
            label=labels.get(method, method),
        )

    title = dataset.attrs.get("title", "Ceilometer")
    ax.set_title(f"{title}: ABLH from {variable}")
    ax.set_xlabel("UTC time")
    ax.set_ylabel("Range above instrument [km]")
    ax.set_ylim(min_range / 1000.0, max_range / 1000.0)
    ax.grid(True, alpha=0.25)
    ax.legend(loc="upper right")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))

    cbar = fig.colorbar(mesh, ax=ax)
    cbar.set_label(f"log10({variable})")

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=160)
    plt.close(fig)


def find_file_for_date(directory: Path, date: str, *, pattern: str = "*.nc") -> Path:
    """Find one NetCDF file for a date.

    ``date`` may be written as ``YYYY-MM-DD`` or ``YYYYMMDD``. The search is
    recursive, so it also looks inside subfolders.
    """
    directory = Path(directory).expanduser()
    date_token = date.replace("-", "")
    matches = sorted(directory.rglob(f"*{date_token}*{pattern.replace('*', '')}"))

    if not matches:
        raise FileNotFoundError(f"No file found for date {date} in {directory}")
    if len(matches) > 1:
        print(f"WARNING: several files found for {date}; using the first one:")
        for match in matches[:5]:
            print(f"  - {match}")
        if len(matches) > 5:
            print(f"  ... and {len(matches) - 5} more")

    return matches[0]


def run_ceilometer_ablh_workbench(
    *,
    raw_path: Path,
    cloudnet_path: Path,
    output_dir: Path,
    raw_variable: str = "beta_raw",
    cloudnet_variable: str = "beta",
    time_frequency: str | None = None,
    min_range: float = 500.0,
    max_range: float = 4000.0,
    wct_threshold: float = 0.05,
    wct_width: float = 300.0,
    temporal_threshold: float = 1e-5,
    temporal_window_minutes: float = 10.0,
    apply_cloudnet_mask: bool = True,
    run_label: str | None = None,
) -> tuple[dict[str, Path], Path]:
    """Run WCT and temporal-variance ABLH detection for one day.

    The ABLH is calculated from the raw file. If ``apply_cloudnet_mask=True``,
    Cloudnet is only used to remove noisy raw pixels.
    """
    raw_path = Path(raw_path).expanduser()
    cloudnet_path = Path(cloudnet_path).expanduser()
    output_dir = Path(output_dir).expanduser()
    output_dir.mkdir(parents=True, exist_ok=True)

    label = run_label or raw_path.stem
    frequency_label = time_frequency or "native"
    range_label = f"r{min_range:g}-{max_range:g}m"

    print(f"Loading raw data: {raw_path.name}")
    raw_dataset = load_backscatter_for_ablh(
        raw_path,
        variable=raw_variable,
        time_frequency=time_frequency,
    )

    if apply_cloudnet_mask:
        print(f"Loading Cloudnet mask: {cloudnet_path.name}")
        cloudnet_dataset = load_backscatter_for_ablh(
            cloudnet_path,
            variable=cloudnet_variable,
            time_frequency=None,
        )
        working_dataset = apply_cloudnet_noise_mask(
            raw_dataset,
            cloudnet_dataset,
            raw_variable=raw_variable,
            cloudnet_variable=cloudnet_variable,
        )
    else:
        working_dataset = raw_dataset

    if not np.isfinite(working_dataset[raw_variable]).any():
        raise ValueError(
            f"No finite values remain in {raw_variable!r}. "
            "Check the input files and the Cloudnet mask."
        )

    method_options = {
        "wct": {
            "threshold": wct_threshold,
            "wct_width": wct_width,
            "time_window_minutes": temporal_window_minutes,
        },
        "temporal_variance": {
            "threshold": temporal_threshold,
            "wct_width": wct_width,
            "time_window_minutes": temporal_window_minutes,
        },
    }

    output_paths: dict[str, Path] = {}
    results: dict[str, xr.Dataset] = {}

    for method in METHODS:
        print(f"Running ABLH method: {method}")
        ablh_result = detect_ablh(
            working_dataset,
            input_kind="cloudnet",
            variable=raw_variable,
            method=method,
            min_range=min_range,
            max_range=max_range,
            **method_options[method],
        )

        ablh_result.attrs.update(
            {
                "source_raw_file": str(raw_path),
                "source_cloudnet_file": str(cloudnet_path),
                "raw_variable": raw_variable,
                "cloudnet_variable_used_as_mask": cloudnet_variable if apply_cloudnet_mask else "not_used",
                "time_frequency": frequency_label,
                "min_range_m": float(min_range),
                "max_range_m": float(max_range),
                "wct_threshold": float(wct_threshold),
                "wct_width_m": float(wct_width),
                "temporal_threshold": float(temporal_threshold),
                "temporal_window_minutes": float(temporal_window_minutes),
            }
        )

        output_path = output_dir / f"{label}_{method}_{raw_variable}_{frequency_label}_{range_label}_ablh.nc"
        ablh_result.to_netcdf(output_path)
        output_paths[method] = output_path
        results[method] = ablh_result

    quicklook_path = output_dir / f"quicklook_{label}_{raw_variable}_{frequency_label}_{range_label}_ablh.png"
    plot_ablh_quicklook(
        dataset=working_dataset,
        ablh_results=results,
        variable=raw_variable,
        min_range=min_range,
        max_range=max_range,
        output_path=quicklook_path,
    )

    print("Done.")
    return output_paths, quicklook_path

## 3. General Variables and Parameters

This is the section you normally edit.

Adjust `main_dir`, `raw_dir`, `cloudnet_dir`, and `target_date` to match your data. If the raw and Cloudnet files are in the same folder, you can set `raw_dir = main_dir` and `cloudnet_dir = main_dir`.

Tip: use this notebook when you want to inspect or tune one date before running the batch notebook.

In [ ]:
# Main project folder.
main_dir = Path(r"C:\Users\usuario\Documents\github\abl_detection")

# Folders where files are searched recursively.
raw_dir = main_dir / "data" / "raw"
cloudnet_dir = main_dir / "data" / "cloudnet"

# Folder where ABLH NetCDF files and quicklook figures are saved.
output_dir = main_dir / "ablh_outputs"

# Date to process. Accepted formats are YYYY-MM-DD or YYYYMMDD.
target_date = "2023-08-02"

# Variables inside the NetCDF files.
raw_variable = "beta_raw"
cloudnet_variable = "beta"

# Vertical range used to search for the ABLH, in metres above the instrument.
min_range = 500.0
max_range = 4000.0

# Keep None to preserve the original raw temporal resolution.
# Alternative example: "5min", but that smooths/aggregates the temporal signal.
time_frequency = None

# WCT method parameters.
wct_threshold = 0.05
wct_width = 300.0

# temporal_variance method parameters.
temporal_threshold = 1e-5
temporal_window_minutes = 10.0

# If True, use Cloudnet as a noise mask on top of the raw data.
apply_cloudnet_mask = True

### Quick Path Check

This cell does not process anything yet. It only helps you check whether the configured folders exist.

If any path prints `False`, review the path before running the final processing loop.

In [ ]:
print(f"main_dir exists:     {main_dir.exists()} -> {main_dir}")
print(f"raw_dir exists:      {raw_dir.exists()} -> {raw_dir}")
print(f"cloudnet_dir exists: {cloudnet_dir.exists()} -> {cloudnet_dir}")
print(f"output_dir:          {output_dir}")

## 4. Single-Date Processing

This cell processes the date configured in `target_date`.

It finds the matching raw and Cloudnet files, runs both ABLH methods, and saves the NetCDF outputs plus a quicklook figure.

In [ ]:
print("=" * 80)
print(f"Processing date: {target_date}")

raw_path = find_file_for_date(raw_dir, target_date)
cloudnet_path = find_file_for_date(cloudnet_dir, target_date)

print(f"Raw file:      {raw_path}")
print(f"Cloudnet file: {cloudnet_path}")

date_label = target_date.replace("-", "")
ablh_paths, quicklook_path = run_ceilometer_ablh_workbench(
    raw_path=raw_path,
    cloudnet_path=cloudnet_path,
    output_dir=output_dir,
    raw_variable=raw_variable,
    cloudnet_variable=cloudnet_variable,
    time_frequency=time_frequency,
    min_range=min_range,
    max_range=max_range,
    wct_threshold=wct_threshold,
    wct_width=wct_width,
    temporal_threshold=temporal_threshold,
    temporal_window_minutes=temporal_window_minutes,
    apply_cloudnet_mask=apply_cloudnet_mask,
    run_label=date_label,
)

single_output = {
    "target_date": target_date,
    "ablh_paths": ablh_paths,
    "quicklook_path": quicklook_path,
}

single_output